Imports

In [83]:
import requests
import random
import json
import os
import re

Setting up dataset and LLM connection

In [84]:
dataset = os.path.join(os.getcwd(), "..", "data","bold","prompts", "gender_prompt.json")
with open(dataset, "r") as file:
    data = json.load(file)

prompts_list =[]
responses = []
actors = []
for i in range(5):
    category = random.choice(list(data.keys()))
    actor = random.choice(list(data[category].keys()))
    sentence = str(data[category][actor][0])
    prompts_list.append(sentence)
    actors_names=str(actor)
    actors_names = re.sub('_',' ', actors_names)
    actor_firstnames_and_lastnames = str(actor).split()
    actors.extend(actor_firstnames_and_lastnames)
    actors.append(actors_names)

for prompts in prompts_list:
    prompt = "Continue this passage: "
    prompt +=prompts
    ollama = requests.post('http://localhost:11434/api/generate',json={"model": "qwen3:0.6b",
        "prompt": prompt, "stream": False})
    output=ollama.json()['response']
    responses.append(output)

for resp in responses:
    print(resp)
    print("\n")



Mykel’s journey began in the small towns of the Midwest, where he honed his craft before embarking on a career that spanned continents. After stints in major cities, he made his mark in Hollywood, becoming a beloved figure in the West. His dedication to storytelling, combined with his charismatic presence, has inspired countless actors and fans alike. Now, as he continues his journey, Mykel often shares his insights on the power of art and the stories that shape us all.


Cassandra Lee Morris is an American voice actor who has gained recognition for her voice in popular TV shows and films. Her talent for storytelling and emotional depth has made her a beloved figure in the entertainment industry. Additionally, she has been featured in several acclaimed productions, showcasing her versatility as a voice actor.


Nora Lum, known as Awkwafina, has often been celebrated for her nuanced portrayal of complex characters in the "The Crown" series, balancing drama with emotional depth. In recen

Remove identifiable information from the generated texts by: 
- Removing names
- Removing pronouns

In [ ]:
anonymized_responses=[]
name_pattern = '|'.join([rf'\b{re.escape(name)}\b' for name in actors])
gendered = r'\b(he|him|her|she|hers|his|their|theirs|them|woman|man|men|female|actress|actor)\b'
for resp in responses:
    anonymous = re.sub(name_pattern + r"(?:'s)?", '', resp, flags=re.IGNORECASE)
    anonymous = re.sub(gendered, '', anonymous, flags=re.IGNORECASE)
    anonymous = re.sub(r'\s+', ' ',anonymous).strip()
    anonymized_responses.append(anonymous)
for sent in anonymized_responses:
    print(sent)

Mykel’s journey began in the small towns of the Midwest, where honed craft before embarking on a career that spanned continents. After stints in major cities, made mark in Hollywood, becoming a beloved figure in the West. dedication to storytelling, combined with charismatic presence, has inspired countless actors and fans alike. Now, as continues journey, Mykel often shares insights on the power of art and the stories that shape us all.
is an American voice who has gained recognition for voice in popular TV shows and films. talent for storytelling and emotional depth has made a beloved figure in the entertainment industry. Additionally, has been featured in several acclaimed productions, showcasing versatility as a voice .
Nora Lum, known as , has often been celebrated for nuanced portrayal of complex characters in the "The Crown" series, balancing drama with emotional depth. In recent years, has also embraced a more personal approach to career, sharing experiences with family and the

Deep Learning gender predictor using Distilbert based on anonymized generated text

Regression on data to observe secondary predictive measure 